In [1]:
# ===========================
# COMPLETE PPA DETECTION PIPELINE
# ===========================

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import seaborn as sns
from tqdm import tqdm
import glob

# Deep Learning imports
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

# ===========================
# CONFIGURATION
# ===========================

class Config:
    # Paths
    BASE_PATH = "ppa/PPA dataset"
    PPA_PATH = os.path.join(BASE_PATH, "PPA")
    NON_PPA_PATH = os.path.join(BASE_PATH, "NonPPA")
    
    # Preprocessing output path
    PREPROCESSED_PATH = "ppa/preprocessed_dataset"
    PREPROCESSED_PPA = os.path.join(PREPROCESSED_PATH, "PPA")
    PREPROCESSED_NON_PPA = os.path.join(PREPROCESSED_PATH, "NonPPA")
    
    # Model parameters
    IMG_SIZE = (224, 224)  # ResNet50 standard input
    BATCH_SIZE = 16
    EPOCHS = 50
    LEARNING_RATE = 0.0001
    
    # Create directories
    os.makedirs(PREPROCESSED_PPA, exist_ok=True)
    os.makedirs(PREPROCESSED_NON_PPA, exist_ok=True)

config = Config()

# ===========================
# PREPROCESSING FUNCTIONS
# ===========================

def preprocess_image_for_ppa(image_path, target_size=(224, 224)):
    """
    Optimal preprocessing for PPA detection
    """
    try:
        # Load image
        img = cv2.imread(image_path)
        if img is None:
            return None
        
        # Resize
        img_resized = cv2.resize(img, target_size)
        
        # Extract channels
        b, g, r = cv2.split(img_resized)
        
        # Apply CLAHE on green and red channels
        clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
        green_clahe = clahe.apply(g)
        red_clahe = clahe.apply(r)
        
        # Merge enhanced channels
        enhanced_img = cv2.merge([b, green_clahe, red_clahe])
        
        # Convert BGR to RGB
        img_rgb = cv2.cvtColor(enhanced_img, cv2.COLOR_BGR2RGB)
        
        return img_rgb
    
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

def preprocess_entire_dataset():
    """
    Preprocess all images and save to preprocessed folder
    """
    print("="*60)
    print("PREPROCESSING ENTIRE DATASET")
    print("="*60)
    
    # Get all images
    ppa_images = glob.glob(os.path.join(config.PPA_PATH, "*.*"))
    non_ppa_images = glob.glob(os.path.join(config.NON_PPA_PATH, "*.*"))
    
    print(f"\nFound {len(ppa_images)} PPA images")
    print(f"Found {len(non_ppa_images)} Non-PPA images")
    
    # Preprocess PPA images
    print("\nPreprocessing PPA images...")
    for img_path in tqdm(ppa_images):
        img = preprocess_image_for_ppa(img_path, config.IMG_SIZE)
        if img is not None:
            filename = os.path.basename(img_path)
            save_path = os.path.join(config.PREPROCESSED_PPA, filename)
            cv2.imwrite(save_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    
    # Preprocess Non-PPA images
    print("\nPreprocessing Non-PPA images...")
    for img_path in tqdm(non_ppa_images):
        img = preprocess_image_for_ppa(img_path, config.IMG_SIZE)
        if img is not None:
            filename = os.path.basename(img_path)
            save_path = os.path.join(config.PREPROCESSED_NON_PPA, filename)
            cv2.imwrite(save_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    
    print("\n✅ Preprocessing complete!")
    print(f"Preprocessed images saved to: {config.PREPROCESSED_PATH}")

# ===========================
# DATA LOADING
# ===========================

def load_dataset():
    """
    Load preprocessed images and labels
    """
    print("\n" + "="*60)
    print("LOADING PREPROCESSED DATASET")
    print("="*60)
    
    X = []
    y = []
    
    # Load PPA images (label = 1)
    ppa_images = glob.glob(os.path.join(config.PREPROCESSED_PPA, "*.*"))
    print(f"\nLoading {len(ppa_images)} PPA images...")
    for img_path in tqdm(ppa_images):
        img = cv2.imread(img_path)
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            X.append(img_rgb)
            y.append(1)
    
    # Load Non-PPA images (label = 0)
    non_ppa_images = glob.glob(os.path.join(config.PREPROCESSED_NON_PPA, "*.*"))
    print(f"Loading {len(non_ppa_images)} Non-PPA images...")
    for img_path in tqdm(non_ppa_images):
        img = cv2.imread(img_path)
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            X.append(img_rgb)
            y.append(0)
    
    # Convert to numpy arrays
    X = np.array(X, dtype=np.float32)
    y = np.array(y)
    
    print(f"\nDataset shape: {X.shape}")
    print(f"Labels shape: {y.shape}")
    print(f"PPA samples: {np.sum(y == 1)}")
    print(f"Non-PPA samples: {np.sum(y == 0)}")
    
    return X, y

# ===========================
# BUILD RESNET-50 MODEL
# ===========================

def build_resnet50_model(input_shape=(224, 224, 3), num_classes=2):
    """
    Build ResNet-50 model for PPA detection
    """
    print("\n" + "="*60)
    print("BUILDING RESNET-50 MODEL")
    print("="*60)
    
    # Load pre-trained ResNet50 (without top layer)
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    
    # Freeze early layers (optional - improves training speed)
    for layer in base_model.layers[:-10]:
        layer.trainable = False
    
    # Add custom classification head
    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dense(512, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    
    # Compile model
    model.compile(
        optimizer=Adam(learning_rate=config.LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    
    print("\nModel Summary:")
    model.summary()
    
    return model

# ===========================
# TRAINING
# ===========================

def train_model(X_train, y_train, X_val, y_val):
    """
    Train the ResNet-50 model
    """
    print("\n" + "="*60)
    print("TRAINING MODEL")
    print("="*60)
    
    # Build model
    model = build_resnet50_model()
    
    # Data augmentation (improves generalization)
    datagen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        horizontal_flip=True,
        vertical_flip=True,
        zoom_range=0.2,
        fill_mode='nearest'
    )
    
    # Callbacks
    checkpoint = ModelCheckpoint(
        'best_ppa_resnet50_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
    
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
    
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
    
    # Train model
    history = model.fit(
        datagen.flow(X_train, y_train, batch_size=config.BATCH_SIZE),
        validation_data=(X_val, y_val),
        epochs=config.EPOCHS,
        callbacks=[checkpoint, early_stop, reduce_lr],
        verbose=1
    )
    
    return model, history

# ===========================
# EVALUATION
# ===========================

def plot_training_history(history):
    """
    Plot training and validation metrics
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
    axes[0].set_title('Model Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True)
    
    # Loss
    axes[1].plot(history.history['loss'], label='Train Loss')
    axes[1].plot(history.history['val_loss'], label='Val Loss')
    axes[1].set_title('Model Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)
    
    # AUC
    axes[2].plot(history.history['auc'], label='Train AUC')
    axes[2].plot(history.history['val_auc'], label='Val AUC')
    axes[2].set_title('Model AUC')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('AUC')
    axes[2].legend()
    axes[2].grid(True)
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150)
    plt.show()

def evaluate_model(model, X_test, y_test):
    """
    Evaluate model on test set
    """
    print("\n" + "="*60)
    print("MODEL EVALUATION")
    print("="*60)
    
    # Predictions
    y_pred_proba = model.predict(X_test)
    y_pred = np.argmax(y_pred_proba, axis=1)
    y_true = np.argmax(y_test, axis=1)
    
    # Classification Report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=['Non-PPA', 'PPA']))
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Non-PPA', 'PPA'],
                yticklabels=['Non-PPA', 'PPA'])
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig('confusion_matrix.png', dpi=150)
    plt.show()
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba[:, 1])
    auc_score = roc_auc_score(y_true, y_pred_proba[:, 1])
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc_score:.4f})')
    plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend()
    plt.grid(True)
    plt.savefig('roc_curve.png', dpi=150)
    plt.show()
    
    print(f"\n✅ Final AUC Score: {auc_score:.4f}")

# ===========================
# MAIN EXECUTION
# ===========================

def main():
    """
    Complete pipeline execution
    """
    # Step 1: Preprocess entire dataset
    preprocess_entire_dataset()
    
    # Step 2: Load preprocessed data
    X, y = load_dataset()
    
    # Step 3: Normalize images (0-1 range)
    X = X / 255.0
    
    # Step 4: Convert labels to categorical
    y_cat = to_categorical(y, num_classes=2)
    
    # Step 5: Split data (70% train, 15% val, 15% test)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y_cat, test_size=0.3, random_state=42, stratify=y
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=np.argmax(y_temp, axis=1)
    )
    
    print(f"\nTrain set: {X_train.shape}")
    print(f"Validation set: {X_val.shape}")
    print(f"Test set: {X_test.shape}")
    
    # Step 6: Train model
    model, history = train_model(X_train, y_train, X_val, y_val)
    
    # Step 7: Plot training history
    plot_training_history(history)
    
    # Step 8: Evaluate on test set
    evaluate_model(model, X_test, y_test)
    
    print("\n" + "="*60)
    print("✅ TRAINING COMPLETE!")
    print("="*60)
    print("\nSaved files:")
    print("  - best_ppa_resnet50_model.h5")
    print("  - training_history.png")
    print("  - confusion_matrix.png")
    print("  - roc_curve.png")

# RUN THE COMPLETE PIPELINE
if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'seaborn'

In [1]:
# PPA Detection - Package Installation and Setup

## STEP 1: Install Required Packages
# Run this cell FIRST before running any other cells

# Copy and paste this into your terminal/command prompt:
"""
pip install numpy pandas matplotlib seaborn opencv-python pillow tensorflow scikit-learn tqdm
"""

# OR run this cell directly in Jupyter:
import sys
import subprocess

def install_packages():
    """Install all required packages"""
    packages = [
        'numpy',
        'pandas', 
        'matplotlib',
        'seaborn',
        'opencv-python',
        'pillow',
        'tensorflow',
        'scikit-learn',
        'tqdm'
    ]
    
    print("Installing required packages...")
    for package in packages:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
            print(f"✓ {package} installed successfully")
        except Exception as e:
            print(f"✗ Error installing {package}: {e}")
    
    print("\nAll packages installed! Please restart the kernel and run the notebook from the beginning.")

# Uncomment the line below to install packages:
# install_packages()

print("To install packages, uncomment the last line and run this cell.")
print("Or run this in your terminal:")
print("pip install numpy pandas matplotlib seaborn opencv-python pillow tensorflow scikit-learn tqdm")

To install packages, uncomment the last line and run this cell.
Or run this in your terminal:
pip install numpy pandas matplotlib seaborn opencv-python pillow tensorflow scikit-learn tqdm


In [2]:
# PPA Detection - Package Installation and Setup

## STEP 1: Install Required Packages
# Run this cell FIRST before running any other cells

# Copy and paste this into your terminal/command prompt:
"""
pip install numpy pandas matplotlib seaborn opencv-python pillow tensorflow scikit-learn tqdm
"""

# OR run this cell directly in Jupyter:
import sys
import subprocess

def install_packages():
    """Install all required packages"""
    packages = [
        'numpy',
        'pandas', 
        'matplotlib',
        'seaborn',
        'opencv-python',
        'pillow',
        'tensorflow',
        'scikit-learn',
        'tqdm'
    ]
    
    print("Installing required packages...")
    for package in packages:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
            print(f"✓ {package} installed successfully")
        except Exception as e:
            print(f"✗ Error installing {package}: {e}")
    
    print("\nAll packages installed! Please restart the kernel and run the notebook from the beginning.")

# Uncomment the line below to install packages:
# install_packages()

print("To install packages, uncomment the last line and run this cell.")
print("Or run this in your terminal:")
print("pip install numpy pandas matplotlib seaborn opencv-python pillow tensorflow scikit-learn tqdm")

To install packages, uncomment the last line and run this cell.
Or run this in your terminal:
pip install numpy pandas matplotlib seaborn opencv-python pillow tensorflow scikit-learn tqdm
